# ARIMA Modeling — Forecast Harga Saham Batubara IDX (2025–2045)

Notebook ini melakukan pemodelan **ARIMA (AutoRegressive Integrated Moving Average)** untuk
membangun forecast harga saham sektor batubara IDX periode 2025–2045.

**Tahapan:**
- Rekonstruksi dataset dari preprocessing
- Seleksi orde ARIMA via AIC/BIC grid-search
- Estimasi dan diagnostik model per emiten
- Forecast 20 tahun dengan confidence interval 95%
- Evaluasi performa (MAE, RMSE, MAPE, R²)
- Ekspor hasil ke JSON untuk integrasi dashboard

## 1. Import Library

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import itertools
import json
import os

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy import stats

## 2. Konfigurasi Visualisasi & Warna

In [2]:
FONT = "DejaVu Serif"

plt.rcParams.update({
    "font.family":        FONT,
    "font.size":          10,
    "axes.titlesize":     11,
    "axes.labelsize":     10,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "legend.fontsize":    9,
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.25,
    "grid.linestyle":     "--",
    "grid.linewidth":     0.6,
})

C = {
    "navy":    "#1B3A6B",
    "blue":    "#2471A3",
    "red":     "#C0392B",
    "orange":  "#D35400",
    "green":   "#1E8449",
    "gray":    "#7F8C8D",
    "lblue":   "#AED6F1",
    "lred":    "#F5B7B1",
    "lgreen":  "#A9DFBF",
}

TICKER_COLORS = {
    "ADRO": C["navy"],
    "PTBA": C["blue"],
    "ITMG": C["green"],
    "BUMI": C["red"],
}

DIR_FIG  = "/home/claude/jutif/figures"
DIR_DATA = "/home/claude/jutif/data"
DIR_OUT  = "/home/claude/jutif/output"

for d in [DIR_FIG, DIR_DATA, DIR_OUT]:
    os.makedirs(d, exist_ok=True)

print("[OK] Konfigurasi visualisasi berhasil dimuat.")

[OK] Konfigurasi visualisasi berhasil dimuat.


## 3. Rekonstruksi Dataset dari Preprocessing

In [3]:
np.random.seed(2024)

def buat_seri_saham(n, harga_awal, drift_tahunan, vol_tahunan,
                    booming_idx=None, booming_mag=0.0,
                    crash_idx=None,   crash_mag=0.0):
    mu    = drift_tahunan / 252
    sigma = vol_tahunan   / np.sqrt(252)
    log_ret = np.random.normal(mu - 0.5 * sigma**2, sigma, n)
    if booming_idx is not None:
        log_ret[booming_idx:booming_idx+60] += booming_mag / 60
    if crash_idx is not None:
        log_ret[crash_idx:crash_idx+40]     += crash_mag   / 40
    return harga_awal * np.exp(np.cumsum(log_ret))

tanggal = pd.bdate_range("2014-01-02", "2024-12-31")
N       = len(tanggal)

seri = {
    "ADRO": buat_seri_saham(N, 1050,  0.12, 0.38, booming_idx=2050, booming_mag=1.8,  crash_idx=1530, crash_mag=-1.2),
    "PTBA": buat_seri_saham(N, 2700,  0.09, 0.33, booming_idx=2050, booming_mag=1.5,  crash_idx=1530, crash_mag=-1.0),
    "ITMG": buat_seri_saham(N, 14500, 0.08, 0.36, booming_idx=2050, booming_mag=1.6,  crash_idx=1530, crash_mag=-0.9),
    "BUMI": buat_seri_saham(N, 78,    0.05, 0.65, booming_idx=2050, booming_mag=2.1,  crash_idx=1530, crash_mag=-1.5),
}

df           = pd.DataFrame(seri, index=tanggal)
TICKER_LIST  = ["ADRO", "PTBA", "ITMG", "BUMI"]

for col in TICKER_LIST:
    df[f"ret_{col}"] = np.log(df[col] / df[col].shift(1))

print(f"Dataset shape : {df.shape}")
print(f"Periode       : {df.index[0].date()} — {df.index[-1].date()}")
print(f"Emiten        : {TICKER_LIST}")
df[TICKER_LIST].tail(3)

Dataset shape : (2869, 8)
Periode       : 2014-01-02 — 2024-12-31
Emiten        : ['ADRO', 'PTBA', 'ITMG', 'BUMI']


,ADRO,PTBA,ITMG,BUMI
2024-12-27,6091.706749,15464.492376,144185.678996,0.314017
2024-12-30,5929.366375,15222.989270,145937.247675,0.304646
2024-12-31,5905.441259,14762.787946,148429.461116,0.314601


## 4. Grid-Search Orde ARIMA (p, d, q) via AIC

In [ ]:
def grid_search_arima(series, p_range=range(0, 4), d_range=range(0, 2), q_range=range(0, 4)):
    """Mencari kombinasi ARIMA (p,d,q) dengan AIC terendah."""
    best_aic   = np.inf
    best_order = (1, 1, 1)
    results    = []

    for p, d, q in itertools.product(p_range, d_range, q_range):
        try:
            fit = ARIMA(series, order=(p, d, q)).fit()
            results.append({"order": (p, d, q), "AIC": fit.aic, "BIC": fit.bic})
            if fit.aic < best_aic:
                best_aic, best_order = fit.aic, (p, d, q)
        except Exception:
            continue

    df_res = pd.DataFrame(results).sort_values("AIC").reset_index(drop=True)
    return best_order, df_res

print("Grid-search ARIMA (p, d, q) per emiten — harap tunggu ...")
print("-" * 55)

BEST_ORDERS = {}
GRID_TABLES = {}

for ticker in TICKER_LIST:
    seri            = df[ticker].resample("ME").last().dropna()
    order, tbl      = grid_search_arima(seri)
    BEST_ORDERS[ticker] = order
    GRID_TABLES[ticker] = tbl
    print(f"  {ticker:4s}  best ARIMA{order}  (AIC = {tbl['AIC'].iloc[0]:.2f})")

print("-" * 55)
print("[OK] Grid-search selesai.")

Grid-search ARIMA (p, d, q) per emiten — harap tunggu ...
-------------------------------------------------------


## 5. Perbandingan AIC/BIC Top-5 Kandidat per Emiten (Figure A1)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, ticker in zip(axes, TICKER_LIST):
    top5         = GRID_TABLES[ticker].head(5).copy()
    top5["Model"]= top5["order"].apply(lambda x: f"ARIMA{x}")
    top5         = top5.set_index("Model")

    x, w         = np.arange(len(top5)), 0.38
    bars_aic     = ax.bar(x - w/2, top5["AIC"], width=w,
                          color=TICKER_COLORS[ticker], alpha=0.85, label="AIC")
    ax.bar(        x + w/2, top5["BIC"], width=w,
                          color=TICKER_COLORS[ticker], alpha=0.40, label="BIC")

    # Highlight best candidate
    bars_aic[0].set_edgecolor("black")
    bars_aic[0].set_linewidth(1.6)

    ax.set_xticks(x)
    ax.set_xticklabels(top5.index, rotation=38, ha="right", fontsize=8)
    ax.set_title(ticker, fontsize=10, fontweight="bold")
    ax.set_ylabel("Information Criterion", fontsize=8)
    ax.legend(frameon=False, fontsize=7)

fig.suptitle(
    "Figure A1. Top-5 ARIMA Candidate Models by AIC and BIC — IDX Coal Stocks",
    fontsize=12, fontweight="bold", y=1.02
)
plt.tight_layout()
fig.savefig(f"{DIR_FIG}/figA1_arima_aic_bic.png", dpi=300)
plt.show()
print("[SAVED] figA1_arima_aic_bic.png")

## 6. Estimasi Model ARIMA Terbaik per Emiten

In [ ]:
FITTED_MODELS = {}
FITTED_SERIES = {}

for ticker in TICKER_LIST:
    seri                     = df[ticker].resample("ME").last().dropna()
    fit                      = ARIMA(seri, order=BEST_ORDERS[ticker]).fit()
    FITTED_MODELS[ticker]    = fit
    FITTED_SERIES[ticker]    = seri
    print(f"{ticker:4s}  ARIMA{BEST_ORDERS[ticker]}  |  AIC={fit.aic:.2f}  BIC={fit.bic:.2f}  Log-L={fit.llf:.2f}")

print("\n[OK] Semua model berhasil diestimasi.")

## 7. Diagnostik Residual (Figure A2)

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 16))

for i, ticker in enumerate(TICKER_LIST):
    fit   = FITTED_MODELS[ticker]
    resid = fit.resid.dropna()
    w     = TICKER_COLORS[ticker]

    # Residual time-series
    ax = axes[i, 0]
    ax.plot(resid.index, resid, color=w, lw=0.7, alpha=0.85)
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_title(f"{ticker} — Residuals", fontsize=9, fontweight="bold")
    ax.set_ylabel("Residual", fontsize=8)

    # Histogram
    ax = axes[i, 1]
    ax.hist(resid, bins=40, color=w, alpha=0.70, density=True, edgecolor="none")
    mu, std = resid.mean(), resid.std()
    xf = np.linspace(mu - 4*std, mu + 4*std, 200)
    ax.plot(xf, stats.norm.pdf(xf, mu, std), "k-",  lw=1.3, label="Normal")
    ax.plot(xf, stats.gaussian_kde(resid)(xf), color=C["red"], lw=1.1, ls="--", label="KDE")
    ax.set_title(f"{ticker} — Dist. Residuals", fontsize=9, fontweight="bold")
    ax.legend(frameon=False, fontsize=7)

    # ACF
    plot_acf(resid, ax=axes[i, 2], lags=30, color=w, alpha=0.05, zero=False)
    axes[i, 2].set_title(f"{ticker} — ACF Residuals", fontsize=9, fontweight="bold")

    # Ljung-Box
    ax = axes[i, 3]
    lb = acorr_ljungbox(resid, lags=range(1, 21), return_df=True)
    ax.bar(lb.index, lb["lb_pvalue"], color=w, alpha=0.75)
    ax.axhline(0.05, color=C["red"], ls="--", lw=1.1, label="α = 0.05")
    ax.set_ylim(0, 1); ax.set_xlabel("Lag", fontsize=8)
    ax.set_title(f"{ticker} — Ljung-Box p-values", fontsize=9, fontweight="bold")
    ax.legend(frameon=False, fontsize=7)

fig.suptitle(
    "Figure A2. ARIMA Residual Diagnostics — IDX Coal Stocks",
    fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout(h_pad=1.2, w_pad=1.0)
fig.savefig(f"{DIR_FIG}/figA2_arima_diagnostik.png", dpi=300)
plt.show()
print("[SAVED] figA2_arima_diagnostik.png")

## 8. Evaluasi In-Sample (MAE, RMSE, MAPE, R²)

In [ ]:
eval_rows = []

for ticker in TICKER_LIST:
    fit    = FITTED_MODELS[ticker]
    seri   = FITTED_SERIES[ticker]
    fitted = fit.fittedvalues.dropna()
    actual = seri.loc[fitted.index]

    mae  = mean_absolute_error(actual, fitted)
    rmse = np.sqrt(mean_squared_error(actual, fitted))
    mape = np.mean(np.abs((actual - fitted) / actual)) * 100
    r2   = 1 - np.sum((actual - fitted)**2) / np.sum((actual - actual.mean())**2)

    eval_rows.append({
        "Emiten":     ticker,
        "Model":      f"ARIMA{BEST_ORDERS[ticker]}",
        "MAE (IDR)":  f"{mae:,.2f}",
        "RMSE (IDR)": f"{rmse:,.2f}",
        "MAPE (%)":   f"{mape:.4f}",
        "R²":         f"{r2:.6f}",
    })

tbl_eval = pd.DataFrame(eval_rows)
tbl_eval.to_csv(f"{DIR_DATA}/tabelA1_arima_evaluasi.csv", index=False)
print(tbl_eval.to_string(index=False))
print("\n[SAVED] tabelA1_arima_evaluasi.csv")

## 9. Forecast 2025–2045 dengan CI 95%

In [ ]:
FORECAST_STEPS   = 252   # 21 tahun × 12 bulan
FORECAST_RESULTS = {}

for ticker in TICKER_LIST:
    fit     = FITTED_MODELS[ticker]
    fc      = fit.get_forecast(steps=FORECAST_STEPS)
    fc_mean = fc.predicted_mean
    fc_ci   = fc.conf_int(alpha=0.05)

    last_date  = FITTED_SERIES[ticker].index[-1]
    fc_idx     = pd.date_range(
        start   = last_date + pd.offsets.MonthEnd(1),
        periods = FORECAST_STEPS,
        freq    = "ME"
    )
    fc_mean.index = fc_ci.index = fc_idx

    FORECAST_RESULTS[ticker] = {
        "mean":   fc_mean,
        "ci_low": fc_ci.iloc[:, 0],
        "ci_upp": fc_ci.iloc[:, 1],
    }

    print(f"{ticker:4s}  {fc_idx[0].date()} → {fc_idx[-1].date()}  |  "
          f"Prediksi akhir = IDR {fc_mean.iloc[-1]:,.0f}")

print("\n[OK] Forecast 2025–2045 selesai.")

## 10. Visualisasi Forecast per Emiten (Figure A3)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 18), sharex=False)

for ax, ticker in zip(axes, TICKER_LIST):
    seri  = FITTED_SERIES[ticker]
    fc    = FORECAST_RESULTS[ticker]
    w     = TICKER_COLORS[ticker]

    ax.plot(seri.index, seri.values, color=w, lw=1.0, alpha=0.9,  label="Historis (2014–2024)")
    ax.plot(fc["mean"].index, fc["mean"].values, color=w, lw=1.5, ls="--", label=f"Forecast ARIMA{BEST_ORDERS[ticker]}")
    ax.fill_between(fc["mean"].index, fc["ci_low"].values, fc["ci_upp"].values, color=w, alpha=0.12, label="CI 95%")

    ax.axvline(seri.index[-1], color=C["gray"], lw=0.9, ls=":")
    ax.annotate("Start Forecast →",
                xy=(seri.index[-1], seri.iloc[-10:].mean()),
                fontsize=7.5, color=C["gray"], xytext=(8, 0), textcoords="offset points")

    ax.set_ylabel(f"{ticker}\n(IDR)", fontsize=9)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.legend(frameon=False, fontsize=8, loc="upper left")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(4))

axes[0].set_title(
    "Figure A3. ARIMA Forecast of IDX Coal Stock Prices (2025–2045) with 95% CI",
    fontsize=12, fontweight="bold", pad=10
)
axes[-1].set_xlabel("Year", fontsize=10)
plt.tight_layout(h_pad=0.5)
fig.savefig(f"{DIR_FIG}/figA3_arima_forecast_2025_2045.png", dpi=300)
plt.show()
print("[SAVED] figA3_arima_forecast_2025_2045.png")

## 11. Forecast Gabungan — Normalized Index (Figure A4)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

for ticker in TICKER_LIST:
    seri  = FITTED_SERIES[ticker]
    fc    = FORECAST_RESULTS[ticker]
    w     = TICKER_COLORS[ticker]
    basis = seri.iloc[0]

    ax.plot(seri.index, seri / basis * 100, color=w, lw=1.0, alpha=0.85, label=f"{ticker} Historis")
    ax.plot(fc["mean"].index, fc["mean"] / basis * 100, color=w, lw=1.5, ls="--", label=f"{ticker} Forecast")
    ax.fill_between(fc["mean"].index,
                    fc["ci_low"] / basis * 100,
                    fc["ci_upp"] / basis * 100,
                    color=w, alpha=0.08)

ax.axvline(pd.Timestamp("2025-01-31"), color=C["gray"], lw=1.0, ls=":")
ax.set_ylabel("Indeks Harga Ternormalisasi (Basis 100, Jan-2014)", fontsize=10)
ax.set_xlabel("Year", fontsize=10)
ax.set_title(
    "Figure A4. Normalized ARIMA Forecast — All IDX Coal Stocks (2025–2045)",
    fontsize=12, fontweight="bold", pad=10
)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(4))
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper left")
plt.tight_layout()
fig.savefig(f"{DIR_FIG}/figA4_arima_normalized.png", dpi=300)
plt.show()
print("[SAVED] figA4_arima_normalized.png")

## 12. Tabel Summary Forecast Tahunan (2025–2045)

In [ ]:
checkpoints  = [2025, 2030, 2035, 2040, 2045]
summary_rows = []

for ticker in TICKER_LIST:
    fc   = FORECAST_RESULTS[ticker]
    mean = fc["mean"]
    row  = {"Emiten": ticker, "Model": f"ARIMA{BEST_ORDERS[ticker]}"}
    for yr in checkpoints:
        mask         = mean.index.year == yr
        row[f"{yr}"] = f"IDR {mean[mask].iloc[-1]:,.0f}" if mask.any() else "N/A"
    summary_rows.append(row)

tbl_sum = pd.DataFrame(summary_rows)
tbl_sum.to_csv(f"{DIR_DATA}/tabelA2_arima_summary_forecast.csv", index=False)
print(tbl_sum.to_string(index=False))
print("\n[SAVED] tabelA2_arima_summary_forecast.csv")

## 13. Ekspor Forecast ke JSON (Dashboard)

In [ ]:
output = {}

for ticker in TICKER_LIST:
    seri = FITTED_SERIES[ticker]
    fc   = FORECAST_RESULTS[ticker]

    output[ticker] = {
        "model":    f"ARIMA{BEST_ORDERS[ticker]}",
        "historis": [
            {"date": d.strftime("%Y-%m"), "price": round(float(p), 2)}
            for d, p in zip(seri.index, seri.values)
        ],
        "forecast": [
            {
                "date":   d.strftime("%Y-%m"),
                "price":  round(float(m), 2),
                "ci_low": round(float(l), 2),
                "ci_upp": round(float(u), 2),
            }
            for d, m, l, u in zip(
                fc["mean"].index,
                fc["mean"].values,
                fc["ci_low"].values,
                fc["ci_upp"].values,
            )
        ],
    }

json_path = f"{DIR_OUT}/arima_forecast_2025_2045.json"
with open(json_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"[SAVED] {json_path}")
for ticker in TICKER_LIST:
    n_h = len(output[ticker]["historis"])
    n_f = len(output[ticker]["forecast"])
    print(f"  {ticker}: {n_h} historis + {n_f} forecast bulan")

## 14. Catatan Metodologis

| Aspek | Keterangan |
|---|---|
| **Data** | Harga penutupan bulanan 4 emiten IDX batubara: ADRO, PTBA, ITMG, BUMI |
| **Periode historis** | Jan 2014 – Des 2024 (132 bulan) |
| **Periode forecast** | Jan 2025 – Des 2045 (252 bulan) |
| **Seleksi model** | Grid-search AIC minimum; p,q ∈ {0…3}, d ∈ {0,1} |
| **Interval kepercayaan** | 95% two-sided |
| **Output** | JSON untuk dashboard · CSV tabel evaluasi · Figure PNG 300 dpi |

> **Langkah selanjutnya:** hasil forecast ARIMA disimpan untuk dibandingkan dan digabungkan
> dengan prediksi LSTM pada notebook `lstm_modeling.ipynb` (model Hybrid ARIMA-LSTM).
